<a href="https://colab.research.google.com/github/nileve-teo/INSTRUMENTACION-BIOMEDICA/blob/main/Simulaci%C3%B3n_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Laboratorio N.° 3**:

Simulador de *sEMG*

In [7]:
!pip install wfdb

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

# 1. Concatenación y escalamiento a 8 bits de la señal
senal_completa = np.concatenate([
    senales["Reposo"],
    senales["Cierre de mano"],
    senales["Extensión del índice"]
])

senal_min = np.min(senal_completa)
senal_max = np.max(senal_completa)

senal_8bits = np.round(
    (senal_completa - senal_min) * 255 / (senal_max - senal_min)
).astype(np.uint8)

# 2. Configuración y Conversión Digital a Tensión
mpl.rcParams["animation.embed_limit"] = 50.0

FS_SEMG = 2048
V_REF = 3.3
tension_dac = (senal_8bits.astype(float) / 255) * V_REF

# Tiempos de las 3 etapas
t_reposo = len(senales["Reposo"]) / FS_SEMG
t_cierre = len(senales["Cierre de mano"]) / FS_SEMG
t_extension = len(senales["Extensión del índice"]) / FS_SEMG

t_corte1 = t_reposo
t_corte2 = t_reposo + t_cierre
t_total = t_corte2 + t_extension

# Parámetros de animación (Osciloscopio)
T_INICIO_VENTANA = 0.5
T_FIN_VENTANA = 2.0
VENTANA = T_FIN_VENTANA - T_INICIO_VENTANA
FPS = 30
DURACION = 15  # segundos de simulación
N_VENTANA = int(VENTANA * FS_SEMG)

# 3. Creación de la figura con 2 SUBPLOTS
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={'height_ratios': [2, 1]})

# --- Subplot 1: Osciloscopio Virtual (Ventana ampliada) ---
(linea_osc,) = ax1.plot([], [], linewidth=1, color="black")
ax1.set_xlim(T_INICIO_VENTANA, T_FIN_VENTANA)
ax1.set_ylim(-0.5, 4.0)
ax1.set_ylabel("Tensión estimada (V)")
ax1.set_title("Osciloscopio Virtual - Detección de Etapas sEMG")
ax1.grid(True, linestyle="--", alpha=0.5)

indicador_tiempo = ax1.text(0.02, 0.90, "", transform=ax1.transAxes, fontsize=10)
indicador_etapa = ax1.text(
    0.02, 0.78, "", transform=ax1.transAxes, fontsize=11, fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8)
)

# --- Subplot 2: Señal Completa y Posición Actual ---
tiempo_global = np.arange(len(tension_dac)) / FS_SEMG
ax2.plot(tiempo_global, tension_dac, color="gray", linewidth=0.6, alpha=0.7, label="Señal Completa")

# Sombras estáticas de etapas en la señal global
ax2.axvspan(0, t_corte1, color="lightblue", alpha=0.3, label="Reposo")
ax2.axvspan(t_corte1, t_corte2, color="lightgreen", alpha=0.3, label="Cierre de mano")
ax2.axvspan(t_corte2, t_total, color="salmon", alpha=0.3, label="Extensión del índice")

# Marcador móvil (línea roja)
linea_cursor = ax2.axvline(x=0, color="red", linestyle="--", linewidth=1.5, label="Posición Actual")

# Variable global para guardar la región sombreada actual
region_ventana = None

ax2.set_xlim(0, t_total)
ax2.set_ylim(-0.5, 4.0)
ax2.set_xlabel("Tiempo Global (s)")
ax2.set_ylabel("Tensión (V)")
ax2.set_title("Ubicación en Tiempo Real sobre la Señal sEMG Completa")
ax2.legend(loc="upper right", fontsize=8, ncol=5)
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()

# Control de etapas y animación
def obtener_etapa(t):
    t_mod = t % t_total
    if t_mod < t_corte1:
        return "Etapa: Reposo", "lightblue"
    elif t_mod < t_corte2:
        return "Etapa: Cierre de mano", "lightgreen"
    else:
        return "Etapa: Extensión del índice", "salmon"

def inicializar():
    global region_ventana
    linea_osc.set_data([], [])
    indicador_tiempo.set_text("")
    indicador_etapa.set_text("")
    ax1.set_facecolor("white")
    linea_cursor.set_xdata([0])
    if region_ventana is not None:
        region_ventana.remove()
        region_ventana = None
    return linea_osc, indicador_tiempo, indicador_etapa, linea_cursor

def actualizar(frame):
    global region_ventana
    inicio = int(frame * FS_SEMG / FPS)
    fin = inicio + N_VENTANA
    indices = np.arange(inicio, fin) % len(tension_dac)
    t_ventana = np.arange(N_VENTANA) / FS_SEMG + T_INICIO_VENTANA

    # 1. Actualizar Osciloscopio (Arriba)
    linea_osc.set_data(t_ventana, tension_dac[indices])

    t_actual = (inicio / FS_SEMG) % t_total
    etapa_nombre, color_fondo = obtener_etapa(t_actual)
    ax1.set_facecolor(color_fondo)

    indicador_tiempo.set_text(f"Tiempo de reproducción: {t_actual:.2f} s")
    indicador_etapa.set_text(etapa_nombre)

    # 2. Actualizar Rotulado y Cursor en Señal Completa (Abajo)
    linea_cursor.set_xdata([t_actual])

    # Eliminar sombreado anterior y dibujar el nuevo
    if region_ventana is not None:
        region_ventana.remove()

    t_fin_rotulo = min(t_actual + VENTANA, t_total)
    region_ventana = ax2.axvspan(t_actual, t_fin_rotulo, color="red", alpha=0.25)

    return linea_osc, indicador_tiempo, indicador_etapa, linea_cursor

# Generar Animación
total_frames = int(DURACION * FPS)
animacion = FuncAnimation(
    fig,
    actualizar,
    init_func=inicializar,
    frames=total_frames,
    interval=1000 / FPS,
    blit=False,
)

plt.close(fig)
display(HTML(animacion.to_jshtml()))

In [15]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation

# 1. Concatenación y escalamiento a 8 bits de la señal
senal_completa = np.concatenate([
    senales["Reposo"],
    senales["Cierre de mano"],
    senales["Extensión del índice"]
])

senal_min = np.min(senal_completa)
senal_max = np.max(senal_completa)

senal_8bits = np.round(
    (senal_completa - senal_min) * 255 / (senal_max - senal_min)
).astype(np.uint8)

# 2. Configuración y Parámetros
FS_SEMG = 2048
V_REF = 3.3
tension_dac = (senal_8bits.astype(float) / 255) * V_REF

# Tiempos de las 3 etapas
t_reposo = len(senales["Reposo"]) / FS_SEMG
t_cierre = len(senales["Cierre de mano"]) / FS_SEMG
t_extension = len(senales["Extensión del índice"]) / FS_SEMG

t_corte1 = t_reposo
t_corte2 = t_reposo + t_cierre
t_total = t_corte2 + t_extension

# Parámetros para la generación del GIF
T_INICIO_VENTANA = 0.5
T_FIN_VENTANA = 2.0
VENTANA = T_FIN_VENTANA - T_INICIO_VENTANA
FPS = 20
DURACION = 15
N_VENTANA = int(VENTANA * FS_SEMG)

# 3. Creación de la figura
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5.5), gridspec_kw={'height_ratios': [2, 1]})

# --- Subplot 1: Osciloscopio ---
(linea_osc,) = ax1.plot([], [], linewidth=1, color="black")
ax1.set_xlim(T_INICIO_VENTANA, T_FIN_VENTANA)
ax1.set_ylim(-0.5, 4.0)
ax1.set_ylabel("Tensión estimada (V)")
ax1.set_title("Osciloscopio Virtual - Detección de Etapas sEMG")
ax1.grid(True, linestyle="--", alpha=0.5)

indicador_tiempo = ax1.text(0.02, 0.88, "", transform=ax1.transAxes, fontsize=10)
indicador_etapa = ax1.text(
    0.02, 0.74, "", transform=ax1.transAxes, fontsize=10, fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8)
)

# --- Subplot 2: Señal Completa ---
tiempo_global = np.arange(len(tension_dac)) / FS_SEMG
ax2.plot(tiempo_global, tension_dac, color="gray", linewidth=0.5, alpha=0.7)

ax2.axvspan(0, t_corte1, color="lightblue", alpha=0.3, label="Reposo")
ax2.axvspan(t_corte1, t_corte2, color="lightgreen", alpha=0.3, label="Cierre de mano")
ax2.axvspan(t_corte2, t_total, color="salmon", alpha=0.3, label="Extensión del índice")

linea_cursor = ax2.axvline(x=0, color="red", linestyle="--", linewidth=1.5)
region_ventana = None

ax2.set_xlim(0, t_total)
ax2.set_ylim(-0.5, 4.0)
ax2.set_xlabel("Tiempo Global (s)")
ax2.set_ylabel("Tensión (V)")
ax2.set_title("Ubicación en Tiempo Real sobre la Señal sEMG Completa")
ax2.legend(loc="upper right", fontsize=8, ncol=4)
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()

def obtener_etapa(t):
    t_mod = t % t_total
    if t_mod < t_corte1:
        return "Etapa: Reposo", "lightblue"
    elif t_mod < t_corte2:
        return "Etapa: Cierre de mano", "lightgreen"
    else:
        return "Etapa: Extensión del índice", "salmon"

def inicializar():
    global region_ventana
    linea_osc.set_data([], [])
    indicador_tiempo.set_text("")
    indicador_etapa.set_text("")
    ax1.set_facecolor("white")
    linea_cursor.set_xdata([0])
    if region_ventana is not None:
        region_ventana.remove()
        region_ventana = None
    return linea_osc, indicador_tiempo, indicador_etapa, linea_cursor

def actualizar(frame):
    global region_ventana
    inicio = int(frame * FS_SEMG / FPS)
    fin = inicio + N_VENTANA
    indices = np.arange(inicio, fin) % len(tension_dac)
    t_ventana = np.arange(N_VENTANA) / FS_SEMG + T_INICIO_VENTANA

    # 1. Actualización del osciloscopio
    linea_osc.set_data(t_ventana, tension_dac[indices])
    t_actual = (inicio / FS_SEMG) % t_total
    etapa_nombre, color_fondo = obtener_etapa(t_actual)
    ax1.set_facecolor(color_fondo)

    indicador_tiempo.set_text(f"Tiempo de reproducción: {t_actual:.2f} s")
    indicador_etapa.set_text(etapa_nombre)

    # 2. Actualización de la posición en la señal completa
    linea_cursor.set_xdata([t_actual])
    if region_ventana is not None:
        region_ventana.remove()

    t_fin_rotulo = min(t_actual + VENTANA, t_total)
    region_ventana = ax2.axvspan(t_actual, t_fin_rotulo, color="red", alpha=0.25)

    return linea_osc, indicador_tiempo, indicador_etapa, linea_cursor

# 4. Compilación y exportación a archivo GIF
print("Generando archivo GIF, esto tomará unos segundos...")
total_frames = int(DURACION * FPS)
animacion = FuncAnimation(
    fig,
    actualizar,
    init_func=inicializar,
    frames=total_frames,
    interval=1000 / FPS,
    blit=False,
)

# Guardar GIF sin intentar desplegar render HTML en el navegador
animacion.save("simulacion_semg.gif", writer="pillow", fps=FPS)
plt.close(fig)

print("¡Proceso completado! 'simulacion_semg.gif' ya está listo en el panel de archivos de Colab.")

Generando archivo GIF, esto tomará unos segundos...
¡Proceso completado! 'simulacion_semg.gif' ya está listo en el panel de archivos de Colab.
